In [24]:
#Library
import pandas as pd
import matplotlib.pyplot as plt
import spacy
from collections import Counter
import sys
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from preprocessing import preprocessing


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder



from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import spacy.cli
spacy.cli.download("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 99.4 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [25]:
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

In [26]:
df_train=pd.read_csv("data/train.csv")
df_test=pd.read_csv("data/test.csv")
df_train=df_train.drop(columns=['id'])
df_train=df_train.drop_duplicates()

df_train=df_train.drop(columns='location')
df_test=df_test.drop(columns='location')


In [27]:
ponctuation = [".", "!", "?"]
preprocesser=FunctionTransformer(preprocessing)

In [28]:
X_train=df_train.drop(columns="target")
y_train=df_train["target"]

In [29]:
columns=["has_.","has_?","has_!","has_url","has_CAP","url_is_https"]

features = ColumnTransformer([
    ("txt", TfidfVectorizer(), "texte_clean"),                    # ← la colonne nettoyée
    ("kw",  OneHotEncoder(handle_unknown="ignore"), ["keyword"]),
    ("binaire", "passthrough", columns),
])

pipe = Pipeline([
    ("preprocesser", preprocesser),   # nettoyage seulement
    ("features", features),             # remplace l'étape "vect"
    ("clf", LogisticRegression(max_iter=1000)),
])


param_grid = [
    {"features": [CountVectorizer(), TfidfVectorizer()],
     "features__ngram_range": [(1, 1), (1, 2)],
     "features__min_df": [1, 3]},
]

In [30]:
pipe.fit(X_train.head(50), y_train.head(50))

KeyError: "['location'] not found in axis"

In [ ]:
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro",n_jobs=-1)

In [ ]:
print("X_train :", X_train.shape)
out = pipe.named_steps["preprocesser"].transform(X_train)
print("après preprocessing :", out.shape, type(out))

X_train : (7561, 3)
après preprocessing : (7561, 10) <class 'pandas.DataFrame'>


In [ ]:
print(out.columns.tolist())

['keyword', 'text', 'has_.', 'has_!', 'has_?', 'has_url', 'url_is_https', 'has_CAP', 'tokens', 'texte_clean']


In [ ]:
grid.fit(X_train,y_train)

ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
8 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/python/lib/python3.13/site-packages/sklearn/pipeline.py", line 649, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 140

In [ ]:
y_test_pred=grid.predict(df_test)

submission = pd.DataFrame({
    "id":df_test["id"],
    "target":y_test_pred
})

submission.to_csv("submission_pipeline.csv",index=False)
